## Preprocessing METABRIC non-omics data
- Demographics
- Clinical
- Paraclinical

In [2]:
import pandas as pd
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.impute import SimpleImputer

In [ ]:
data = pd.read_csv("D:/NCKH/CURE/data/metabric/brca_metabric_clinical_data.tsv", sep="\t")

In [9]:
data.shape

(2509, 39)

In [10]:
data = data.rename(columns={'Sample ID': 'SAMPLE_ID'})

In [11]:
drop_cols = ['Study ID', 'Sex', 'Patient ID', 'Sample Type',
             'Cancer Type', 'Number of Samples Per Patient', 
             "Patient's Vital Status"]

In [12]:
data = data.drop(drop_cols, axis=1)

In [13]:
print(data.isna().sum().sort_values(ascending=False))

3-Gene classifier subtype         745
Tumor Stage                       721
Primary Tumor Laterality          639
Cellularity                       592
Type of Breast Surgery            554
HER2 status measured by SNP6      529
Radio Therapy                     529
PR Status                         529
Inferred Menopausal State         529
Hormone Therapy                   529
HER2 Status                       529
Integrative Cluster               529
Chemotherapy                      529
Pam50 + Claudin-low subtype       529
Overall Survival (Months)         528
Overall Survival Status           528
Lymph nodes examined positive     266
Nottingham prognostic index       222
Mutation Count                    151
Tumor Size                        149
Tumor Other Histologic Subtype    135
Relapse Free Status (Months)      121
Neoplasm Histologic Grade         121
ER status measured by IHC          83
ER Status                          40
Relapse Free Status                21
Cohort      

In [14]:
data = data.dropna(subset=['PR Status'])

In [15]:
data.shape

(1980, 32)

In [16]:
print(data.isna().sum().sort_values(ascending=False))

Tumor Stage                       514
3-Gene classifier subtype         216
Mutation Count                    121
Primary Tumor Laterality          110
Neoplasm Histologic Grade          88
Lymph nodes examined positive      76
Cellularity                        63
Tumor Other Histologic Subtype     44
ER status measured by IHC          43
Tumor Size                         26
Type of Breast Surgery             25
Relapse Free Status                 1
Nottingham prognostic index         0
Relapse Free Status (Months)        0
PR Status                           0
Overall Survival Status             0
Overall Survival (Months)           0
Oncotree Code                       0
TMB (nonsynonymous)                 0
Radio Therapy                       0
SAMPLE_ID                           0
Age at Diagnosis                    0
Inferred Menopausal State           0
Hormone Therapy                     0
HER2 Status                         0
HER2 status measured by SNP6        0
ER Status   

# Numerical

In [17]:
missing_cols = [col for col in data.columns if data[col].isna().sum() > 0]
missing_cols

['Type of Breast Surgery',
 'Cellularity',
 'ER status measured by IHC',
 'Neoplasm Histologic Grade',
 'Tumor Other Histologic Subtype',
 'Primary Tumor Laterality',
 'Lymph nodes examined positive',
 'Mutation Count',
 'Relapse Free Status',
 '3-Gene classifier subtype',
 'Tumor Size',
 'Tumor Stage']

In [18]:
s = data[missing_cols].dtypes != 'object'
missing_num_cols = list(s[s].index)

missing_num_cols

['Neoplasm Histologic Grade',
 'Lymph nodes examined positive',
 'Mutation Count',
 'Tumor Size',
 'Tumor Stage']

In [19]:
num_imputer = SimpleImputer(strategy='median')

In [20]:
data[missing_num_cols] = num_imputer.fit_transform(data[missing_num_cols])

# Categorical
## Ordinal Encode

Ordinal encode those that has two categories or the order of categories has meaning. We will one-hot the rest.

In [21]:
s = data.dtypes == 'object'
cat_cols = list(s[s].index)
cat_cols.remove('SAMPLE_ID')
cat_cols

['Type of Breast Surgery',
 'Cancer Type Detailed',
 'Cellularity',
 'Chemotherapy',
 'Pam50 + Claudin-low subtype',
 'ER status measured by IHC',
 'ER Status',
 'HER2 status measured by SNP6',
 'HER2 Status',
 'Tumor Other Histologic Subtype',
 'Hormone Therapy',
 'Inferred Menopausal State',
 'Integrative Cluster',
 'Primary Tumor Laterality',
 'Oncotree Code',
 'Overall Survival Status',
 'PR Status',
 'Radio Therapy',
 'Relapse Free Status',
 '3-Gene classifier subtype']

In [22]:
# Get number of unique entries in each column with categorical data
object_nunique = list(map(lambda col: data[col].nunique(), cat_cols))
d = dict(zip(cat_cols, object_nunique))

# Print number of unique entries by column, in ascending order
sorted(d.items(), key=lambda x: x[1])

[('Type of Breast Surgery', 2),
 ('Chemotherapy', 2),
 ('ER status measured by IHC', 2),
 ('ER Status', 2),
 ('HER2 Status', 2),
 ('Hormone Therapy', 2),
 ('Inferred Menopausal State', 2),
 ('Primary Tumor Laterality', 2),
 ('Overall Survival Status', 2),
 ('PR Status', 2),
 ('Radio Therapy', 2),
 ('Relapse Free Status', 2),
 ('Cellularity', 3),
 ('HER2 status measured by SNP6', 4),
 ('3-Gene classifier subtype', 4),
 ('Pam50 + Claudin-low subtype', 7),
 ('Cancer Type Detailed', 8),
 ('Tumor Other Histologic Subtype', 8),
 ('Oncotree Code', 8),
 ('Integrative Cluster', 11)]

In [23]:
# Columns where order matters! (or just pure binary)
ordinal_cols = [col for col in data.columns if data[col].nunique() == 2]
ordinal_cols.extend(['Cellularity', 'Integrative Cluster'])

ordinal_cols_use_default_encoding = ordinal_cols.copy()
ordinal_cols_use_default_encoding.remove("Cellularity")
ordinal_cols_use_default_encoding

['Type of Breast Surgery',
 'Chemotherapy',
 'ER status measured by IHC',
 'ER Status',
 'HER2 Status',
 'Hormone Therapy',
 'Inferred Menopausal State',
 'Primary Tumor Laterality',
 'Overall Survival Status',
 'PR Status',
 'Radio Therapy',
 'Relapse Free Status',
 'Integrative Cluster']

In [24]:
default_ordinal_encoder = OrdinalEncoder() 
cat_imputer = SimpleImputer(strategy='constant', fill_value=-1)
cell_ordinal_encoder = OrdinalEncoder(categories=[['Low', 'Moderate', 'High']], 
                                      handle_unknown='use_encoded_value',
                                      unknown_value=-1)

In [25]:
data[ordinal_cols_use_default_encoding] = default_ordinal_encoder.fit_transform(data[ordinal_cols_use_default_encoding])
data[ordinal_cols_use_default_encoding] = cat_imputer.fit_transform(data[ordinal_cols_use_default_encoding])

data["Cellularity"] = cell_ordinal_encoder.fit_transform(data["Cellularity"].to_numpy().reshape(-1, 1))

## One-hot encode

In [26]:
# Resetting index to concat with one-hot encoded data later
data = data.reset_index(drop=True)

In [27]:
OH_cols = list(set(cat_cols) - set(ordinal_cols))
OH_cols

['Oncotree Code',
 'Tumor Other Histologic Subtype',
 'Pam50 + Claudin-low subtype',
 'Cancer Type Detailed',
 'HER2 status measured by SNP6',
 '3-Gene classifier subtype']

In [28]:
data[OH_cols].isna().sum().sort_values()

Oncotree Code                       0
Pam50 + Claudin-low subtype         0
Cancer Type Detailed                0
HER2 status measured by SNP6        0
Tumor Other Histologic Subtype     44
3-Gene classifier subtype         216
dtype: int64

In [29]:
OH_imputer = SimpleImputer(strategy='constant')
OH_encoder = OneHotEncoder(sparse_output=False)

In [30]:
data[OH_cols] = OH_imputer.fit_transform(data[OH_cols])
data[OH_cols]

,Oncotree Code,Tumor Other Histologic Subtype,Pam50 + Claudin-low subtype,Cancer Type Detailed,HER2 status measured by SNP6,3-Gene classifier subtype
0,IDC,Ductal/NST,claudin-low,Breast Invasive Ductal Carcinoma,NEUTRAL,ER-/HER2-
1,IDC,Ductal/NST,LumA,Breast Invasive Ductal Carcinoma,NEUTRAL,ER+/HER2- High Prolif
2,IDC,Ductal/NST,LumB,Breast Invasive Ductal Carcinoma,NEUTRAL,missing_value
3,MDLC,Mixed,LumB,Breast Mixed Ductal and Lobular Carcinoma,NEUTRAL,missing_value
4,MDLC,Mixed,LumB,Breast Mixed Ductal and Lobular Carcinoma,NEUTRAL,ER+/HER2- High Prolif
...,...,...,...,...,...,...
1975,ILC,Lobular,LumA,Breast Invasive Lobular Carcinoma,NEUTRAL,ER+/HER2- Low Prolif
1976,IDC,Ductal/NST,LumB,Breast Invasive Ductal Carcinoma,GAIN,missing_value
1977,IDC,Ductal/NST,LumB,Breast Invasive Ductal Carcinoma,NEUTRAL,missing_value
1978,IDC,Ductal/NST,LumB,Breast Invasive Ductal Carcinoma,NEUTRAL,ER+/HER2- High Prolif


In [31]:
encoded_OH_data = pd.DataFrame(OH_encoder.fit_transform(data[OH_cols]))
encoded_OH_data.columns = OH_encoder.get_feature_names_out()
print(encoded_OH_data.dtypes.to_string())

Oncotree Code_BRCA                                                float64
Oncotree Code_BREAST                                              float64
Oncotree Code_IDC                                                 float64
Oncotree Code_ILC                                                 float64
Oncotree Code_IMMC                                                float64
Oncotree Code_MBC                                                 float64
Oncotree Code_MDLC                                                float64
Oncotree Code_PBS                                                 float64
Tumor Other Histologic Subtype_Ductal/NST                         float64
Tumor Other Histologic Subtype_Lobular                            float64
Tumor Other Histologic Subtype_Medullary                          float64
Tumor Other Histologic Subtype_Metaplastic                        float64
Tumor Other Histologic Subtype_Mixed                              float64
Tumor Other Histologic Subtype_Mucinou

In [32]:
data = data.drop(OH_cols, axis=1)

In [33]:
print(data.dtypes.to_string())

SAMPLE_ID                         object
Age at Diagnosis                 float64
Type of Breast Surgery           float64
Cellularity                      float64
Chemotherapy                     float64
Cohort                           float64
ER status measured by IHC        float64
ER Status                        float64
Neoplasm Histologic Grade        float64
HER2 Status                      float64
Hormone Therapy                  float64
Inferred Menopausal State        float64
Integrative Cluster              float64
Primary Tumor Laterality         float64
Lymph nodes examined positive    float64
Mutation Count                   float64
Nottingham prognostic index      float64
Overall Survival (Months)        float64
Overall Survival Status          float64
PR Status                        float64
Radio Therapy                    float64
Relapse Free Status (Months)     float64
Relapse Free Status              float64
TMB (nonsynonymous)              float64
Tumor Size      

In [34]:
data = pd.concat([data, encoded_OH_data], axis=1)

In [35]:
data

,SAMPLE_ID,Age at Diagnosis,Type of Breast Surgery,Cellularity,Chemotherapy,Cohort,ER status measured by IHC,ER Status,Neoplasm Histologic Grade,HER2 Status,...,Cancer Type Detailed_Metaplastic Breast Cancer,HER2 status measured by SNP6_GAIN,HER2 status measured by SNP6_LOSS,HER2 status measured by SNP6_NEUTRAL,HER2 status measured by SNP6_UNDEF,3-Gene classifier subtype_ER+/HER2- High Prolif,3-Gene classifier subtype_ER+/HER2- Low Prolif,3-Gene classifier subtype_ER-/HER2-,3-Gene classifier subtype_HER2+,3-Gene classifier subtype_missing_value
0,MB-0000,75.65,1.0,-1.0,0.0,1.0,1.0,1.0,3.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0
1,MB-0002,43.19,0.0,2.0,0.0,1.0,1.0,1.0,3.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
2,MB-0005,48.87,1.0,2.0,1.0,1.0,1.0,1.0,2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
3,MB-0006,47.68,1.0,1.0,1.0,1.0,1.0,1.0,2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
4,MB-0008,76.97,1.0,2.0,1.0,1.0,1.0,1.0,3.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1975,MB-7295,43.10,0.0,2.0,0.0,4.0,1.0,1.0,3.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
1976,MB-7296,42.88,1.0,2.0,0.0,4.0,1.0,1.0,3.0,1.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,1.0
1977,MB-7297,62.90,1.0,2.0,0.0,4.0,1.0,1.0,3.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0,1.0
1978,MB-7298,61.16,1.0,1.0,0.0,4.0,1.0,1.0,2.0,0.0,...,0.0,0.0,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0


In [36]:
print(data.columns)
print(len(data.columns))

Index(['SAMPLE_ID', 'Age at Diagnosis', 'Type of Breast Surgery',
       'Cellularity', 'Chemotherapy', 'Cohort', 'ER status measured by IHC',
       'ER Status', 'Neoplasm Histologic Grade', 'HER2 Status',
       'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster',
       'Primary Tumor Laterality', 'Lymph nodes examined positive',
       'Mutation Count', 'Nottingham prognostic index',
       'Overall Survival (Months)', 'Overall Survival Status', 'PR Status',
       'Radio Therapy', 'Relapse Free Status (Months)', 'Relapse Free Status',
       'TMB (nonsynonymous)', 'Tumor Size', 'Tumor Stage',
       'Oncotree Code_BRCA', 'Oncotree Code_BREAST', 'Oncotree Code_IDC',
       'Oncotree Code_ILC', 'Oncotree Code_IMMC', 'Oncotree Code_MBC',
       'Oncotree Code_MDLC', 'Oncotree Code_PBS',
       'Tumor Other Histologic Subtype_Ductal/NST',
       'Tumor Other Histologic Subtype_Lobular',
       'Tumor Other Histologic Subtype_Medullary',
       'Tumor Other Histologic

# Post-process and split

In [37]:
# These cols spoil the output
data = data.drop(['Relapse Free Status',
 'Relapse Free Status (Months)'], axis=1)

In [38]:
demographics_cols = [
    "SAMPLE_ID",
    "Age at Diagnosis",
    "Inferred Menopausal State",
    "Cohort",
]

clinical_cols = [
    "SAMPLE_ID",
    "Tumor Stage",
    "Primary Tumor Laterality",
    "Neoplasm Histologic Grade",
    "Tumor Size",
    "Type of Breast Surgery",
    "Nottingham prognostic index",
    'Cancer Type Detailed_Breast',
    'Cancer Type Detailed_Breast Angiosarcoma',
    'Cancer Type Detailed_Breast Invasive Ductal Carcinoma',
    'Cancer Type Detailed_Breast Invasive Lobular Carcinoma',
    'Cancer Type Detailed_Breast Invasive Mixed Mucinous Carcinoma',
    'Cancer Type Detailed_Breast Mixed Ductal and Lobular Carcinoma',
    'Cancer Type Detailed_Invasive Breast Carcinoma',
    'Cancer Type Detailed_Metaplastic Breast Cancer', 'Oncotree Code_BRCA'
]

paraclinical_cols = [
    "SAMPLE_ID",
     '3-Gene classifier subtype_ER+/HER2- High Prolif',
       '3-Gene classifier subtype_ER+/HER2- Low Prolif',
       '3-Gene classifier subtype_ER-/HER2-',
       '3-Gene classifier subtype_HER2+',
       '3-Gene classifier subtype_missing_value',
    "Mutation Count",
    "Lymph nodes examined positive",
    "Cellularity",
    'Tumor Other Histologic Subtype_Ductal/NST',
    'Tumor Other Histologic Subtype_Lobular',
    'Tumor Other Histologic Subtype_Medullary',
    'Tumor Other Histologic Subtype_Metaplastic',
    'Tumor Other Histologic Subtype_Mixed',
    'Tumor Other Histologic Subtype_Mucinous',
    'Tumor Other Histologic Subtype_Other',
    'Tumor Other Histologic Subtype_Tubular/ cribriform',
    'Tumor Other Histologic Subtype_missing_value',
    "ER status measured by IHC",
    "PR Status",
    "HER2 Status",
    'HER2 status measured by SNP6_LOSS',
       'HER2 status measured by SNP6_NEUTRAL',
       'HER2 status measured by SNP6_UNDEF',
    "ER Status",
    "TMB (nonsynonymous)",
    "Integrative Cluster",
    'Pam50 + Claudin-low subtype_Basal', 'Pam50 + Claudin-low subtype_Her2',
       'Pam50 + Claudin-low subtype_LumA', 'Pam50 + Claudin-low subtype_LumB',
       'Pam50 + Claudin-low subtype_NC', 'Pam50 + Claudin-low subtype_Normal',
       'Pam50 + Claudin-low subtype_claudin-low',
    'Oncotree Code_BREAST', 'Oncotree Code_IDC', 'Oncotree Code_ILC',
       'Oncotree Code_IMMC', 'Oncotree Code_MBC', 'Oncotree Code_MDLC',
       'Oncotree Code_PBS', 'HER2 status measured by SNP6_GAIN'
]

treatment_cols = ["SAMPLE_ID", 'Radio Therapy', 'Hormone Therapy', 'Chemotherapy']

output_cols = ["SAMPLE_ID", "Overall Survival Status",
    "Overall Survival (Months)",]

In [39]:
len(demographics_cols) + len(clinical_cols) + len(paraclinical_cols) + len(treatment_cols) + len(output_cols)

69

In [40]:
metabric_demographics = data[demographics_cols]
metabric_clinical = data[clinical_cols]
metabric_paraclinical = data[paraclinical_cols]
metabric_treatment = data[treatment_cols]
metabric_output = data[output_cols]

In [41]:
metabric_output

,SAMPLE_ID,Overall Survival Status,Overall Survival (Months)
0,MB-0000,0.0,140.500000
1,MB-0002,0.0,84.633333
2,MB-0005,1.0,163.700000
3,MB-0006,0.0,164.933333
4,MB-0008,1.0,41.366667
...,...,...,...
1975,MB-7295,0.0,196.866667
1976,MB-7296,1.0,44.733333
1977,MB-7297,1.0,175.966667
1978,MB-7298,1.0,86.233333


In [42]:
data.to_csv('preprocessed_metabric_non_omics.csv', index=False)
metabric_demographics.to_csv('preprocessed_metabric_demographics.csv', index=False)
metabric_clinical.to_csv('preprocessed_metabric_clinical.csv', index=False)
metabric_paraclinical.to_csv('preprocessed_metabric_paraclinical.csv', index=False)
metabric_treatment.to_csv('preprocessed_metabric_treatment.csv', index=False)
metabric_output.to_csv('preprocessed_metabric_output.csv', index=False)